In [0]:
# Create monthly revenue table
from pyspark.sql.functions import (
    sum,
    countDistinct,
    month,
    year,
    col
)

df = spark.table(
    "retail_project_dbricks.default.silver_retail_clean"
)

In [0]:
#  monthly revenue table
monthly_revenue = (
    df.groupBy(
        year("InvoiceDate").alias("Year"),
        month("InvoiceDate").alias("Month"),
        "Country"
    )
    .agg(
        sum("Revenue").alias("TotalRevenue"),
        countDistinct("Invoice").alias("OrderCount")
    )
)

In [0]:
# save monthly revenue table
monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_project_dbricks.default.gold_monthly_revenue"
    )

In [0]:
# top products table
top_products = (
    df.groupBy(
        "StockCode",
        "Description"
    )
    .agg(
        sum("Revenue").alias("TotalRevenue"),
        sum("Quantity").alias("TotalUnits")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
#save top products table
top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_project_dbricks.default.gold_top_products"
    )

In [0]:
#verfy tables 
spark.sql("""
SELECT *
FROM retail_project_dbricks.default.gold_monthly_revenue
LIMIT 10
""").show()
spark.sql("""
SELECT *
FROM retail_project_dbricks.default.gold_top_products
LIMIT 10
""").show()